# Ensembl Genomes — Multi-Kingdom Genome Annotation

[Ensembl Genomes](https://ensemblgenomes.org/) is an ELIXIR Core Data Resource that extends the Ensembl project beyond vertebrates to cover the full breadth of life. It provides genome sequence, gene annotation, comparative genomics, and variation data for thousands of non-vertebrate species through five specialised sub-portals.

| Sub-portal | Kingdom | Example Organism | Approx. Genomes |
|---|---|---|---|
| EnsemblBacteria | Bacteria | *Escherichia coli* K-12 MG1655 | ~50,000 |
| EnsemblFungi | Fungi | *Saccharomyces cerevisiae* S288C | ~1,000 |
| EnsemblMetazoa | Metazoa (non-vertebrate animals) | *Drosophila melanogaster* | ~130 |
| EnsemblPlants | Viridiplantae | *Arabidopsis thaliana* Col-0 | ~120 |
| EnsemblProtists | Protists | *Plasmodium falciparum* 3D7 | ~250 |

**Reference:** Cunningham F. *et al.* (2022). Ensembl 2022. *Nucleic Acids Research*, 50(D1), D988–D995. https://doi.org/10.1093/nar/gkab1049

In [ ]:
import requests
import time
import re
import json
from pathlib import Path

import polars as pl
import pandas as pd

## TODO

### Ingest data
- [x] Connect to Ensembl REST API
- [x] Fetch genome info for representative species across portals
- [x] Download gene annotations for a model organism (Arabidopsis thaliana)

### Explore and clean
- [ ] Gene count distributions across species
- [ ] Biotype breakdown (protein-coding, lncRNA, pseudogene, etc.)
- [ ] Chromosome/scaffold statistics (length distributions, N50)

### Comparative genomics
- [ ] Orthologue counts across kingdoms
- [ ] Synteny analysis introduction

### Functional annotation
- [ ] GO term coverage per species
- [ ] Pathway annotations across species

### Visualization
- [ ] Genome size vs gene count scatter plot
- [ ] Biotype pie charts per species
- [ ] Cross-kingdom ortholog heatmap

### Statistical analysis
- [ ] Gene density distributions
- [ ] Intron/exon statistics
- [ ] Annotation completeness assessment

## 1. Ingest Data

### 1.1 Connect to Ensembl REST API

In [ ]:
ENSEMBL_BASE = "https://rest.ensembl.org"

# Standard headers required by the Ensembl REST API
HEADERS = {"Content-Type": "application/json"}


def ensembl_get(endpoint: str, params: dict | None = None) -> dict | list:
    """Make a GET request to the Ensembl REST API.

    Parameters
    ----------
    endpoint : str
        API endpoint path, e.g. ``"/info/ping"``. Will be appended to
        ``ENSEMBL_BASE``.
    params : dict or None, optional
        Query parameters to include in the request. Defaults to None.

    Returns
    -------
    dict or list
        Parsed JSON response from the API.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{ENSEMBL_BASE}{endpoint}"
    # Be polite: Ensembl allows 15 req/s but a 1 s pause avoids throttling
    time.sleep(1)
    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()
    return response.json()


# --- connectivity check ---
ping = ensembl_get("/info/ping")
print("Ping response:", ping)

software = ensembl_get("/info/software")
print("Ensembl version:", software.get("release"))

### 1.2 Fetch Genome Info for Representative Species

In [ ]:
# Ensure data directory exists
DATA_DIR = Path("/Users/alice/github/elixir-of-life/ensembl-genomes/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Representative species mapped to their Ensembl sub-portal
SPECIES = {
    "arabidopsis_thaliana": "EnsemblPlants",
    "saccharomyces_cerevisiae": "EnsemblFungi",
    "drosophila_melanogaster": "EnsemblMetazoa",
    "escherichia_coli_str_k_12_substr_mg1655": "EnsemblBacteria",
    "homo_sapiens": "Ensembl",
}

ASSEMBLY_FILE = DATA_DIR / "ensembl_genomes_assembly_info.json"

if ASSEMBLY_FILE.exists():
    print(f"Loading cached assembly info from {ASSEMBLY_FILE}")
    with open(ASSEMBLY_FILE) as fh:
        assembly_data = json.load(fh)
else:
    assembly_data = {}
    for species, portal in SPECIES.items():
        print(f"Fetching assembly info for {species} ({portal}) ...")
        info = ensembl_get(f"/info/assembly/{species}")
        # Attach the portal label so it survives serialisation
        info["_portal"] = portal
        assembly_data[species] = info

    # Persist to disk so re-runs are instant
    with open(ASSEMBLY_FILE, "w") as fh:
        json.dump(assembly_data, fh, indent=2)
    print(f"Saved assembly info to {ASSEMBLY_FILE}")

# Summary print
print()
for species, info in assembly_data.items():
    name = info.get("assembly_name", "N/A")
    total_len = info.get("base_pairs", "N/A")
    top_level = len(info.get("top_level_region", []))
    print(f"{species:50s}  assembly={name}  total_bp={total_len:,}  top_level_regions={top_level}")

### 1.3 Download Gene Annotations for Arabidopsis thaliana

In [ ]:
ATHAL_CHR1_FILE = DATA_DIR / "arabidopsis_chr1_genes.json"

# Arabidopsis thaliana chromosome 1 is 30,427,671 bp (TAIR10 assembly)
ATHAL_CHR1_END = 30_427_671

if ATHAL_CHR1_FILE.exists():
    print(f"Loading cached Arabidopsis chr1 genes from {ATHAL_CHR1_FILE}")
    with open(ATHAL_CHR1_FILE) as fh:
        athal_genes = json.load(fh)
else:
    print("Fetching Arabidopsis thaliana chromosome 1 genes ...")
    # The overlap/region endpoint returns all features overlapping a genomic window.
    # content-type must be passed as a query param here because the endpoint also
    # accepts GFF3/BED via the Accept header.
    athal_genes = ensembl_get(
        f"/overlap/region/arabidopsis_thaliana/1:1-{ATHAL_CHR1_END}",
        params={"feature": "gene", "content-type": "application/json"},
    )
    with open(ATHAL_CHR1_FILE, "w") as fh:
        json.dump(athal_genes, fh, indent=2)
    print(f"Saved to {ATHAL_CHR1_FILE}")

print(f"Number of genes fetched on chr1: {len(athal_genes)}")

### 1.4 Parse into DataFrames

In [ ]:
def to_snake(name: str) -> str:
    """Convert a CamelCase or mixed string to snake_case.

    Parameters
    ----------
    name : str
        Input column name in any case convention.

    Returns
    -------
    str
        Normalised snake_case column name.
    """
    # Insert underscore before uppercase letters that follow lowercase letters
    s = re.sub(r"(?<=[a-z0-9])([A-Z])", r"_\1", name)
    return s.lower().replace(" ", "_").replace("-", "_")


# ---------------------------------------------------------------------------
# Assembly info DataFrame
# ---------------------------------------------------------------------------
assembly_rows = []
for species, info in assembly_data.items():
    top_level = info.get("top_level_region", [])
    # Separate chromosomes (coord_system == "chromosome") from scaffolds
    chromosomes = [r for r in top_level if r.get("coord_system") == "chromosome"]
    scaffolds   = [r for r in top_level if r.get("coord_system") != "chromosome"]

    assembly_rows.append({
        "species":          species,
        "portal":           info.get("_portal", ""),
        "assembly_name":    info.get("assembly_name", ""),
        "total_length_bp":  info.get("base_pairs", None),
        "num_chromosomes":  len(chromosomes),
        "num_scaffolds":    len(scaffolds),
    })

df_assembly = pl.DataFrame(assembly_rows)

print("Assembly DataFrame — shape:", df_assembly.shape)
print(df_assembly.head(5))


# ---------------------------------------------------------------------------
# Arabidopsis chr1 genes DataFrame
# ---------------------------------------------------------------------------
gene_rows = []
for gene in athal_genes:
    gene_rows.append({
        "gene_id":     gene.get("id", ""),
        "gene_name":   gene.get("external_name", ""),
        "biotype":     gene.get("biotype", ""),
        "start":       gene.get("start", None),
        "end":         gene.get("end", None),
        "strand":      gene.get("strand", None),
        "description": gene.get("description", ""),
    })

df_genes = pl.DataFrame(gene_rows)

print("\nArabidopsis chr1 genes DataFrame — shape:", df_genes.shape)
print(df_genes.head(5))

## Column Descriptions

### `df_assembly` — one row per representative species

| Column | Type | Description |
|---|---|---|
| `species` | str | Ensembl species identifier (lowercase, underscores) |
| `portal` | str | Ensembl sub-portal the species belongs to (e.g. `EnsemblPlants`) |
| `assembly_name` | str | Official genome assembly name (e.g. `TAIR10`, `GRCh38`) |
| `total_length_bp` | int | Total number of base pairs in the assembly (sum over all sequences) |
| `num_chromosomes` | int | Count of top-level regions with `coord_system == "chromosome"` |
| `num_scaffolds` | int | Count of top-level regions that are not chromosomes (scaffolds, contigs, etc.) |

### `df_genes` — one row per gene on Arabidopsis thaliana chromosome 1

| Column | Type | Description |
|---|---|---|
| `gene_id` | str | Stable Ensembl gene identifier (e.g. `AT1G01010`) |
| `gene_name` | str | Human-readable gene symbol or name if available |
| `biotype` | str | Ensembl biotype classification (e.g. `protein_coding`, `lncRNA`, `pseudogene`) |
| `start` | int | 1-based genomic start coordinate on chromosome 1 |
| `end` | int | 1-based genomic end coordinate on chromosome 1 |
| `strand` | int | Strand of the feature: `1` = forward, `-1` = reverse |
| `description` | str | Free-text functional description from Ensembl/UniProt, may be empty |